# Freddie Mac Mortgage Risk Project
## Notebook 04: Baseline Modeling

This notebook builds the first baseline predictive model for 12-month serious delinquency using the origination-stage Freddie Mac modeling base. The objective is to define an initial feature set, split the data in a time-aware manner, train a benchmark logistic regression model, and evaluate its performance using metrics appropriate for an imbalanced credit-risk classification problem.

## 1. Import packages and define project paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

BASE = Path("/content/drive/MyDrive/freddie_mac_risk_project")
PROCESSED = BASE / "data" / "processed"
TABLES = BASE / "outputs" / "tables"

print("Processed path exists:", PROCESSED.exists())
print("Tables path exists:", TABLES.exists())

Mounted at /content/drive
Processed path exists: True
Tables path exists: True


## 2. Load the modeling base

In [2]:
modeling_base = pd.read_parquet(PROCESSED / "modeling_base_with_target_2019_2023.parquet")

print("Modeling base shape:", modeling_base.shape)
modeling_base.head()

Modeling base shape: (250000, 34)


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_pct,num_units,occupancy_status,cltv,dti,...,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,relief_refinance_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator,vintage_year,target_12m_serious_dq
0,741,2019-03-01,N,2049-02-01,None,0,1,P,80,33,...,Other servicers,None,None,9,None,7,N,7,2019,0
1,731,2019-03-01,N,2049-02-01,13460,0,1,P,77,44,...,Other servicers,None,None,9,None,7,N,7,2019,0
2,722,2019-03-01,N,2049-02-01,None,30,1,P,95,41,...,Other servicers,None,None,9,None,7,N,N,2019,0
3,729,2019-03-01,N,2049-02-01,None,25,1,P,87,17,...,Other servicers,None,None,9,None,7,N,N,2019,0
4,773,2019-03-01,N,2049-02-01,33700,0,1,P,29,43,...,Other servicers,None,None,9,None,7,N,7,2019,0


## 3. Define the initial feature set

In [3]:
baseline_features = [
    "credit_score",
    "first_time_homebuyer_flag",
    "msa",
    "mi_pct",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "channel",
    "property_state",
    "property_type",
    "postal_code",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "servicer_name",
    "program_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator",
    "vintage_year"
]

target_col = "target_12m_serious_dq"

baseline_df = modeling_base[baseline_features + [target_col]].copy()

print("Baseline modeling dataset shape:", baseline_df.shape)
baseline_df.head()

Baseline modeling dataset shape: (250000, 26)


,credit_score,first_time_homebuyer_flag,msa,mi_pct,num_units,occupancy_status,cltv,dti,original_upb,ltv,...,original_loan_term,num_borrowers,seller_name,servicer_name,program_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator,vintage_year,target_12m_serious_dq
0,741,N,None,0,1,P,80,33,372000,80,...,360,2,Other sellers,Other servicers,9,7,N,7,2019,0
1,731,N,13460,0,1,P,77,44,250000,77,...,360,1,Other sellers,Other servicers,9,7,N,7,2019,0
2,722,N,None,30,1,P,95,41,261000,95,...,360,2,Other sellers,Other servicers,9,7,N,N,2019,0
3,729,N,None,25,1,P,87,17,61000,87,...,360,1,Other sellers,Other servicers,9,7,N,N,2019,0
4,773,N,33700,0,1,P,29,43,390000,29,...,360,2,Other sellers,Other servicers,9,7,N,7,2019,0


## 4. Separate predictors and target

In [4]:
X = baseline_df.drop(columns=[target_col])
y = baseline_df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target mean:", y.mean())

X shape: (250000, 25)
y shape: (250000,)
Target mean: 0.010424


The baseline modeling dataset contains 250,000 loans and 25 origination-stage predictor variables. The target rate remains approximately 1.04%, which confirms that the saved modeling base is loading correctly and that the serious delinquency event remains a rare outcome. This reinforces that the task is an imbalanced binary classification problem.

## 5. Create a time-aware train/test split


In [5]:
train_df = baseline_df[baseline_df["vintage_year"].isin([2019, 2020, 2021, 2022])].copy()
test_df = baseline_df[baseline_df["vintage_year"] == 2023].copy()

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nTrain target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (200000, 25) (200000,)
Test shape: (50000, 25) (50000,)

Train target rate: 0.011485
Test target rate: 0.00618


The train/test split is temporally structured, with 2019–2022 used for training and 2023 used as the holdout set. The target rate is lower in the 2023 test cohort than in the earlier training vintages, which is consistent with the vintage-level EDA. This indicates that the model is being evaluated under realistic distribution shift rather than under an easier random split.

## 6. Identify numeric and categorical features

In [6]:
numeric_features = [
    "credit_score",
    "mi_pct",
    "num_units",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "original_loan_term",
    "num_borrowers",
    "property_valuation_method",
    "vintage_year"
]

categorical_features = [col for col in X_train.columns if col not in numeric_features]

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numeric features: ['credit_score', 'mi_pct', 'num_units', 'cltv', 'dti', 'original_upb', 'ltv', 'interest_rate', 'original_loan_term', 'num_borrowers', 'property_valuation_method', 'vintage_year']

Categorical features: ['first_time_homebuyer_flag', 'msa', 'occupancy_status', 'channel', 'property_state', 'property_type', 'postal_code', 'loan_purpose', 'seller_name', 'servicer_name', 'program_indicator', 'interest_only_indicator', 'mi_cancellation_indicator']


The feature split into numeric and categorical variables looks appropriate for a first-pass preprocessing pipeline. Most underwriting ratios and loan size fields are numeric, while geography, product attributes, and seller/servicer descriptors are handled as categorical variables. One field to revisit later is `property_valuation_method`, which is currently treated as numeric but may behave more like a categorical code.

## 7. Build the preprocessing pipeline

In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['credit_score', 'mi_pct', 'num_units', 'cltv',
                                  'dti', 'original_upb', 'ltv', 'interest_rate',
                                  'original_loan_term', 'num_borrowers',
                                  'property_valuation_method',
                                  'vintage_year']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['first_time_homebuyer_flag', 'msa',
                                  'occupancy_status', 'channel',
                                  'property_state', 'property_type',
                                  'postal_code', 'loan_purpose', 'seller_name',
                                  'servicer_name', 'program_indicator',
                                  'interest_only_indicator',
                                  'mi_cancellation_indicator'])])

The preprocessing pipeline is appropriate for a baseline credit-risk model. Numeric variables are median-imputed and standardized, while categorical variables are mode-imputed and one-hot encoded. This produces a reproducible and interview-defensible benchmark without introducing unnecessary complexity at the first modeling stage.

## 8. Train the baseline logistic regression model

In [8]:
logit_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

logit_model.fit(X_train, y_train)
print("Baseline logistic regression model trained successfully.")

Baseline logistic regression model trained successfully.


The baseline logistic regression model trained successfully using the full preprocessing pipeline. This establishes a benchmark model that is both practical and explainable, which is particularly relevant in a credit-risk setting where transparent score behavior matters.

## 9. Evaluate baseline ranking performance

In [9]:
train_pred_proba = logit_model.predict_proba(X_train)[:, 1]
test_pred_proba = logit_model.predict_proba(X_test)[:, 1]

train_roc_auc = roc_auc_score(y_train, train_pred_proba)
test_roc_auc = roc_auc_score(y_test, test_pred_proba)

train_pr_auc = average_precision_score(y_train, train_pred_proba)
test_pr_auc = average_precision_score(y_test, test_pred_proba)

print("Train ROC-AUC:", train_roc_auc)
print("Test ROC-AUC:", test_roc_auc)
print("Train PR-AUC:", train_pr_auc)
print("Test PR-AUC:", test_pr_auc)

Train ROC-AUC: 0.8545708145909493
Test ROC-AUC: 0.6727037818638278
Train PR-AUC: 0.05929857897540857
Test PR-AUC: 0.0191278492434072


The baseline model shows meaningful predictive power, but generalization weakens materially on the 2023 holdout cohort. The drop from train ROC-AUC to test ROC-AUC suggests that the model is capturing real signal but is also sensitive to temporal differences across vintages. The test PR-AUC remains meaningfully above the base event rate, which indicates that the model is still useful as a benchmark in a low-event setting, even though substantial room for improvement remains.

## 10. Review classification output at a simple threshold

In [10]:
test_pred_label = (test_pred_proba >= 0.50).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred_label))

print("\nClassification report:")
print(classification_report(y_test, test_pred_label, digits=4))

Confusion matrix:
[[44850  4841]
 [  223    86]]

Classification report:
              precision    recall  f1-score   support

           0     0.9951    0.9026    0.9466     49691
           1     0.0175    0.2783    0.0328       309

    accuracy                         0.8987     50000
   macro avg     0.5063    0.5904    0.4897     50000
weighted avg     0.9890    0.8987    0.9409     50000



The classification report at a 0.50 threshold should be treated as a provisional diagnostic rather than as the final decision policy. Because the event rate is very low and the model was trained with balanced class weights, the model prioritizes capturing some bad loans at the cost of many false positives. This leads to modest recall for the delinquent class but very low precision, which is typical in an early-stage imbalanced credit-risk model. In later steps, threshold selection should be tied to business trade-offs rather than fixed at 0.50.

## 11. Save baseline model performance summary

In [11]:
baseline_metrics = pd.DataFrame({
    "metric": ["train_roc_auc", "test_roc_auc", "train_pr_auc", "test_pr_auc"],
    "value": [train_roc_auc, test_roc_auc, train_pr_auc, test_pr_auc]
})

baseline_metrics.to_csv(TABLES / "baseline_logit_metrics.csv", index=False)

print(TABLES / "baseline_logit_metrics.csv")
baseline_metrics

/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/baseline_logit_metrics.csv


,metric,value
0,train_roc_auc,0.854571
1,test_roc_auc,0.672704
2,train_pr_auc,0.059299
3,test_pr_auc,0.019128


## Conclusion

A baseline logistic regression model has been trained using origination-stage Freddie Mac features and evaluated on a time-aware holdout set. The model demonstrates meaningful predictive signal, although performance weakens on the most recent vintage cohort, indicating that cohort effects and temporal drift remain important considerations. These baseline results establish a defensible starting benchmark for the project and provide a foundation for threshold tuning, feature refinements, and challenger-model development.